In [3]:
import xarray as xr
import shutil

import os
from os.path import join
import glob
import numpy as np
import datetime

from joblib import Parallel, delayed
import joblib

from glob import glob

from functools import partial
import dask.array as da

import pandas as pd
import tqdm


In [4]:
# top_dir = "/glade/derecho/scratch/dkimpara/goes-cloud-dataset"
top_dir = "/glade/derecho/scratch/dkimpara/goes_10km_train/baselines/climo"

channels = [4, 7, 8, 9, 10, 13]
zarr_path = "/glade/derecho/scratch/dkimpara/goes-cloud-dataset/goes_10km_2025.zarr"

test = 1
num_cpus = 2

In [5]:
zarr_ds = xr.open_dataset(zarr_path, consolidated=False).drop_duplicates(
        dim="t"
    ).sortby(
        "t"
    )

zarr_ds = zarr_ds.sel(t=slice(pd.Timestamp("2025-06-13"), pd.Timestamp("2025-07-09")))
if test:
    print("testing")
    zarr_ds = zarr_ds.isel(t=slice(2, 6))

testing


In [4]:
def compute_std_dev(indices):
    mean_ds = xr.open_dataset(join(top_dir, "climo_mean_2025_test.nc"))
    mean_da = mean_ds["mean"]

    zarr_ds = xr.open_dataset("/glade/derecho/scratch/dkimpara/goes-cloud-dataset/goes_10km.zarr", consolidated=False).drop_duplicates(
        dim="t"
    ).sortby(
        "t"
    )
    zarr_ds = zarr_ds.sel(t=slice(pd.Timestamp("2022-07-01"), pd.Timestamp("2022-12-31")))

    first_index = indices[0]

    ds = zarr_ds.isel(t=first_index)
    sum_square_diff = ((ds.BT_or_R - mean_da) ** 2).fillna(0)

    for i in indices[1:]:
        ds = zarr_ds.isel(t=i)
        sum_square_diff += ((ds.BT_or_R - mean_da) ** 2).fillna(0)

    return sum_square_diff / len(indices)



def minmaxmean(indices):
    zarr_ds = xr.open_dataset("/glade/derecho/scratch/dkimpara/goes-cloud-dataset/goes_10km.zarr", consolidated=False).drop_duplicates(
        dim="t"
    ).sortby(
        "t"
    )
    zarr_ds = zarr_ds.sel(t=slice(pd.Timestamp("2022-07-01"), pd.Timestamp("2022-12-31")))

    first_index = indices[0]
    da = zarr_ds.isel(t=first_index).BT_or_R.fillna(0)
    
    for i in indices[1:]:
        da_2 = zarr_ds.isel(t=i).BT_or_R
        da += da_2.fillna(0)

    means = da / len(indices)

    return means


In [18]:
indices = list(range(len(zarr_ds.t)))
# indices = list(range(2,6))

chunked = np.array_split(indices, num_cpus - 1)

if test:
    chunked = [indices[ :2], indices[2:]]

In [6]:
means = Parallel(n_jobs = num_cpus - 1)(delayed(minmaxmean)(index_list)
                            for index_list in chunked)

mean_da = sum(means) / len(means)
ds = mean_da.to_dataset(name="mean")

ds.to_netcdf(join(top_dir, "climo_mean_2025_test.nc"))

In [7]:
results = Parallel(n_jobs = num_cpus - 1)(delayed(compute_std_dev)(index_list)
                            for index_list in chunked)

ds = xr.open_dataset(join(top_dir, "climo_mean_2025_test.nc"),)

ds["std"] = np.sqrt(sum(results))

ds.to_netcdf(join(top_dir, "data_stats_2025_test.nc"))